# CommGuard detector evaluation v2

Restore an immutable benign archive, print coverage first, and fit only after the strict eight-family/three-run/30-second gate passes. The communication-only block is primary; other ablations are diagnostic.

> **Prototype scope:** CommGuard's Kaggle workflow is a single-node, dual-NVIDIA-T4 research prototype. It validates experimental methodology and software behavior on two local GPU ranks. It does not establish generalization to two physical 8-GPU nodes, NVLink/NVSwitch fabrics, RoCE or InfiniBand networks, large frontier-model workloads, or production treaty-verification deployments.


In [ ]:
import hashlib
import importlib
import os
from pathlib import Path
import re
import subprocess
import sys

NOTEBOOK_VERSION = "commguard_detector_evaluation_v2"
REPOSITORY_URL = "https://github.com/waqasm86/CommGuard.git"
INSTALL_SOURCE = "auto"  # auto: wheel, source archive, pinned Git commit, then dev source.
PINNED_PUBLIC_COMMIT = ""  # Required for public Git installation.
EXPECTED_PACKAGE_SHA256 = ""  # Required for a supplied wheel or source archive.
EXPECTED_NOTEBOOK_SHA256 = ""  # SHA-256 of this canonical source notebook.
DEVELOPMENT_SMOKE_TEST = False
DEVELOPMENT_SOURCE = Path("/kaggle/working/commguard-development-source")
REPOSITORY = Path("/kaggle/working/commguard-source")

def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

wheel_candidates = sorted(Path("/kaggle/input").rglob("commguard*.whl"))
archive_candidates = sorted(
    path for path in Path("/kaggle/input").rglob("commguard*")
    if path.is_file() and path.name.endswith((".tar.gz", ".zip"))
    and not any(token in path.name for token in (
        "-prototype-", "review-bundle", "calibration", "benign-corpus",
        "detector-evaluation", "adversarial-redteam",
    ))
)
selected = None
method = INSTALL_SOURCE
if method == "auto":
    method = "wheel" if wheel_candidates else "archive" if archive_candidates else "git"
if method == "wheel":
    if len(wheel_candidates) != 1:
        raise RuntimeError(f"Expected exactly one CommGuard wheel, observed {wheel_candidates}")
    selected = wheel_candidates[0]
elif method == "archive":
    if len(archive_candidates) != 1:
        raise RuntimeError(
            f"Expected exactly one CommGuard source archive, observed {archive_candidates}"
        )
    selected = archive_candidates[0]

if selected is not None:
    actual_package_sha256 = file_sha256(selected)
    if not re.fullmatch(r"[0-9a-f]{64}", EXPECTED_PACKAGE_SHA256):
        raise RuntimeError("Set EXPECTED_PACKAGE_SHA256 for the supplied package.")
    if actual_package_sha256 != EXPECTED_PACKAGE_SHA256:
        raise RuntimeError("Supplied package SHA-256 does not match.")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--no-deps", str(selected)], check=True
    )
    SOURCE_IDENTITY = f"sha256:{actual_package_sha256}"
    SOURCE_DIRTY = False
    INSTALL_PROVENANCE = {
        "install_source": method,
        "install_path": str(selected),
        "package_sha256": actual_package_sha256,
        "source_identity": SOURCE_IDENTITY,
    }
elif method == "git":
    if not re.fullmatch(r"[0-9a-f]{40}", PINNED_PUBLIC_COMMIT):
        raise RuntimeError("Set PINNED_PUBLIC_COMMIT to a pushed 40-character commit.")
    if not REPOSITORY.exists():
        subprocess.run(
            [
                "git", "clone", "--filter=blob:none", "--no-checkout",
                REPOSITORY_URL, str(REPOSITORY),
            ],
            check=True,
        )
    if not (REPOSITORY / ".git").is_dir():
        raise RuntimeError(f"Refusing non-Git source directory: {REPOSITORY}")
    subprocess.run(
        ["git", "-C", str(REPOSITORY), "fetch", "origin", PINNED_PUBLIC_COMMIT],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(REPOSITORY), "checkout", "--detach", PINNED_PUBLIC_COMMIT], check=True
    )
    head = subprocess.run(
        ["git", "-C", str(REPOSITORY), "rev-parse", "HEAD"], check=True,
        capture_output=True, text=True,
    ).stdout.strip()
    dirty = subprocess.run(
        ["git", "-C", str(REPOSITORY), "status", "--porcelain"], check=True,
        capture_output=True, text=True,
    ).stdout.strip()
    pushed_refs = subprocess.run(
        ["git", "-C", str(REPOSITORY), "branch", "-r", "--contains", head], check=True,
        capture_output=True, text=True,
    ).stdout.strip()
    if head != PINNED_PUBLIC_COMMIT or dirty or not pushed_refs:
        raise RuntimeError("Pinned Git source is dirty, mismatched, or not remote-visible.")
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "--no-build-isolation",
            "--no-deps", str(REPOSITORY),
        ],
        check=True,
    )
    SOURCE_IDENTITY = head
    SOURCE_DIRTY = False
    INSTALL_PROVENANCE = {
        "install_source": "pinned_public_git_commit",
        "repository_url": REPOSITORY_URL,
        "source_identity": head,
        "remote_refs": pushed_refs.splitlines(),
    }
elif method == "development":
    if not DEVELOPMENT_SMOKE_TEST or not (DEVELOPMENT_SOURCE / "pyproject.toml").is_file():
        raise RuntimeError(
            "Editable development source is allowed only for an explicit smoke test."
        )
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "--no-build-isolation",
            "--no-deps", "-e", str(DEVELOPMENT_SOURCE),
        ],
        check=True,
    )
    SOURCE_IDENTITY = "development-editable"
    SOURCE_DIRTY = True
    INSTALL_PROVENANCE = {
        "install_source": "editable_local_development",
        "source_identity": SOURCE_IDENTITY,
        "development_smoke_only": True,
    }
else:
    raise RuntimeError(f"Unsupported INSTALL_SOURCE={method!r}")

importlib.invalidate_caches()
for module_name in [
    name for name in sys.modules if name == "commguard" or name.startswith("commguard.")
]:
    del sys.modules[module_name]
import commguard
REVIEWED_COMMIT = SOURCE_IDENTITY
PIP_FREEZE = subprocess.run(
    [sys.executable, "-m", "pip", "freeze"], check=True, capture_output=True, text=True
).stdout.splitlines()
INSTALL_PROVENANCE.update({
    "commguard_version": commguard.__version__,
    "commguard_import": str(Path(commguard.__file__).resolve()),
    "python_version": sys.version,
    "pip_freeze": PIP_FREEZE,
})
print({
    "source_identity": SOURCE_IDENTITY,
    "source_dirty": SOURCE_DIRTY,
    "installation": INSTALL_PROVENANCE,
})


In [ ]:
from commguard.artifacts import restore_archive, sha256_file

INPUT_ARCHIVE = Path("/kaggle/input/commguard-benign-corpus-prototype/commguard-benign-corpus-prototype-REPLACE.tar.gz")
EXPECTED_INPUT_SHA256 = ""  # Required: SHA-256 printed by the preceding notebook.
ARTIFACTS = Path("/kaggle/working/commguard-artifacts")

if not re.fullmatch(r"[0-9a-f]{64}", EXPECTED_INPUT_SHA256):
    raise RuntimeError("Set EXPECTED_INPUT_SHA256 to the exact 64-character archive hash.")
actual_input_sha256 = sha256_file(INPUT_ARCHIVE)
if actual_input_sha256 != EXPECTED_INPUT_SHA256:
    raise RuntimeError(
        "Input archive hash mismatch: "
        f"expected={EXPECTED_INPUT_SHA256} actual={actual_input_sha256}"
    )
restore_archive(INPUT_ARCHIVE, ARTIFACTS, expected_sha256=EXPECTED_INPUT_SHA256)
print({"restored_archive": str(INPUT_ARCHIVE), "sha256": actual_input_sha256})


In [ ]:
from datetime import datetime, timezone

NOTEBOOK_RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")


import socket
from commguard.environment.preflight import check_environment, summarize_environment
from commguard.provenance import ProvenanceContext

NOTEBOOK_FILENAME = f"{NOTEBOOK_VERSION}.ipynb"
if not re.fullmatch(r"[0-9a-f]{64}", EXPECTED_NOTEBOOK_SHA256):
    raise RuntimeError("Set EXPECTED_NOTEBOOK_SHA256 to the canonical notebook source hash.")
if (REPOSITORY / "notebooks" / NOTEBOOK_FILENAME).is_file():
    actual_notebook_sha256 = file_sha256(REPOSITORY / "notebooks" / NOTEBOOK_FILENAME)
    if actual_notebook_sha256 != EXPECTED_NOTEBOOK_SHA256:
        raise RuntimeError("Canonical notebook SHA-256 does not match the pinned Git source.")
DIRTY_SOURCE_SMOKE_ONLY = bool(SOURCE_DIRTY and DEVELOPMENT_SMOKE_TEST)
if SOURCE_DIRTY and not DIRTY_SOURCE_SMOKE_ONLY:
    raise RuntimeError("Dirty source cannot create accepted research evidence.")
CONTEXT = ProvenanceContext(
    corpus_id=f"corpus-detector-v2-{NOTEBOOK_RUN_ID}",
    experiment_session_id=f"session-detector-v2-{NOTEBOOK_RUN_ID}",
    collection_id=f"collection-detector-v2-{NOTEBOOK_RUN_ID}",
    node_id=socket.gethostname(),
    source_commit=SOURCE_IDENTITY,
    source_dirty=SOURCE_DIRTY,
    notebook_version="commguard_detector_evaluation_v2",
    input_archive_sha256=EXPECTED_INPUT_SHA256,
    random_seed=20260730,
)
ENVIRONMENT = check_environment(strict=True, output=ARTIFACTS, provenance=CONTEXT)
print(summarize_environment(ENVIRONMENT))
print({
    "notebook_run_id": NOTEBOOK_RUN_ID,
    "experiment_session_id": CONTEXT.experiment_session_id,
    "collection_id": CONTEXT.collection_id,
    "corpus_id": CONTEXT.corpus_id,
    "source_commit": CONTEXT.source_commit,
    "source_dirty": CONTEXT.source_dirty,
    "notebook_sha256": EXPECTED_NOTEBOOK_SHA256,
    "development_smoke_only": DIRTY_SOURCE_SMOKE_ONLY,
    "input_archive_sha256": CONTEXT.input_archive_sha256,
})


In [ ]:
import json
from collections import Counter

from commguard.features import (
    PRIMARY_BENIGN_FAMILIES,
    load_extraction_result,
    require_primary_coverage,
)

BENIGN_MATRIX_SUMMARY_PATH = Path("results/matrix-REPLACE.json")
if BENIGN_MATRIX_SUMMARY_PATH.is_absolute() or ".." in BENIGN_MATRIX_SUMMARY_PATH.parts:
    raise RuntimeError("Benign matrix summary path must be artifact-root-relative.")
matrix_summary_path = ARTIFACTS / BENIGN_MATRIX_SUMMARY_PATH
if not matrix_summary_path.is_file():
    raise RuntimeError(f"Exact benign matrix summary is missing: {BENIGN_MATRIX_SUMMARY_PATH}")
BENIGN_MATRIX_SUMMARY = json.loads(matrix_summary_path.read_text(encoding="utf-8"))
BENIGN_EXTRACTION_SUMMARY = Path(BENIGN_MATRIX_SUMMARY["feature_extraction_summary"])
BENIGN_EXTRACTION = load_extraction_result(ARTIFACTS, BENIGN_EXTRACTION_SUMMARY)
if BENIGN_EXTRACTION.calibration_reference != BENIGN_MATRIX_SUMMARY["calibration_reference"]:
    raise RuntimeError("Benign extraction and matrix calibration references differ.")
coverage_by_family = {}
for family in PRIMARY_BENIGN_FAMILIES:
    records = [record for record in BENIGN_EXTRACTION.coverage if record.workload_family == family]
    coverage_by_family[family] = {
        "planned": len(records),
        "included": sum(record.status == "included" for record in records),
        "reason_counts": dict(
            Counter(record.reason_code for record in records if record.reason_code)
        ),
    }
for family, row in sorted(coverage_by_family.items()):
    print({"family": family, **row})
COVERAGE_GATE = require_primary_coverage(
    BENIGN_EXTRACTION,
    required_families=PRIMARY_BENIGN_FAMILIES,
    minimum_runs_per_family=3,
)
print({"coverage_gate": COVERAGE_GATE})


In [ ]:
from commguard.evaluation import evaluate_detector

RUN_MODE = "smoke"  # "smoke" or "full"
if RUN_MODE not in {"smoke", "full"}:
    raise RuntimeError("RUN_MODE must be smoke or full.")
if SOURCE_DIRTY and RUN_MODE != "smoke":
    raise RuntimeError("Dirty editable source is restricted to RUN_MODE='smoke'.")
DEVELOPMENT_SMOKE_ONLY = RUN_MODE == "smoke" or DIRTY_SOURCE_SMOKE_ONLY
RUN_DETECTOR_EVALUATION = True
if not RUN_DETECTOR_EVALUATION:
    raise RuntimeError("Detector evaluation was disabled after the coverage gate.")
EVALUATION = evaluate_detector(
    input_root=ARTIFACTS,
    output=ARTIFACTS,
    required_families=PRIMARY_BENIGN_FAMILIES,
    minimum_runs_per_family=3,
    benign_extraction_summary=BENIGN_EXTRACTION_SUMMARY.relative_to(ARTIFACTS),
)
print({
    "primary_communication_only": EVALUATION["primary_communication_only"],
    "coverage_gate": EVALUATION["coverage_gate"],
    "warnings": EVALUATION["warnings"],
    "exact_calibration_reference": EVALUATION["calibration_reference"],
    "evaluation_artifact_for_next_notebook": EVALUATION["result_artifact"],
})


In [ ]:
from commguard.artifacts import materialize_detector_package

DETECTOR_PACKAGE = materialize_detector_package(
    ARTIFACTS,
    EVALUATION,
    environment=ENVIRONMENT,
    provenance={**CONTEXT.to_dict(), **INSTALL_PROVENANCE},
    notebook_filename=NOTEBOOK_FILENAME,
    notebook_sha256=EXPECTED_NOTEBOOK_SHA256,
    configuration={
        "task": "training_vs_inference",
        "minimum_runs_per_family": 3,
        "primary_feature_set": "communication_only",
        "run_mode": RUN_MODE,
        "development_smoke_only": DEVELOPMENT_SMOKE_ONLY,
    },
)
print({"machine_readable_package": str(DETECTOR_PACKAGE)})


## Results

not executed. No accuracy, robustness, or generalization claim is present.


In [ ]:
from commguard.artifacts import ArtifactStore, sha256_file

ARCHIVE = Path(f"/kaggle/working/commguard-detector-evaluation-prototype-{NOTEBOOK_RUN_ID}.tar.gz")
ARCHIVE, SHA_FILE = ArtifactStore(ARTIFACTS).export_with_checksum(ARCHIVE)
ARCHIVE_SHA256 = sha256_file(ARCHIVE)
print(f"NEXT STEP: add {ARCHIVE} to a private Kaggle dataset without renaming it.")
print(f"NEXT STEP: copy SHA-256 {ARCHIVE_SHA256} into EXPECTED_INPUT_SHA256 in commguard_adversarial_redteam_v1.ipynb.")
print(
    "NEXT STEP: set that notebook's install-source parameters and notebook SHA-256, "
    "then run from the first cell."
)
